# 第二板块衍生：creditos（信贷板块）

本 Notebook 针对原始数据 `衍生原始数据_331条.csv` 中 `response_body.creditos`（信贷账户列表）做特征衍生。

## 你要求的核心点（口径说明）

- **平铺明细**：把每条 credit 账户记录平铺为一行（`apply_id` 会重复），并取出你列出的字段。
- **窗口过滤日期（你说的“查询日期”）**：本板块按你的要求使用 **`fechaAperturaCuenta（开户日期）`** 做窗口过滤。
  - 定义：
    - `days_before_request_credit = request_time - fechaAperturaCuenta`（单位：天）
    - 在每个窗口 N 天内筛选：`0 <= days_before_request_credit <= N`
- **机构归类**：对 `nombreOtorgante` 按你给的字典归 17 类（新增：个征信机构=SIC；不在字典/缺失 -> 其他）。
- **每窗口/每类别输出**：
  - 该类别数量、总数量、该类别/总数量
  - 对你列的每个“计算数据”都计算：**平均值、标准差、最大值、最小值、notNull占比**
- **新增两套同样口径的分组衍生**：
  - `tipoCuenta`：F/H/R/Q/A/E/P（固定付款/抵押贷款/循环信用/无担保信用/授信或预支信用/分期付款信用/质押信用）
  - `tipoCredito`：CC/PP/TC/LC/PB（消费信贷/个人贷款/信用卡/信用额度/个人银行贷款）
- **异常兜底（A/C/D -> -999）**：沿用第一板块规则：
  - A：关键字段缺失（apply_id/request_time/response_body）
  - C：response_body 非法 JSON
  - D：response_body 里没有 creditos（或 creditos 不是 list）
  命中时该 apply_id 的**所有衍生特征**统一填 `-999`。

## 输出（最终只保留 2 个，默认不写，等你确认后再写）

- `outputs/creditos_flat.csv`：creditos 平铺明细（apply_id 重复）
- `outputs/features_creditos.csv`：按 apply_id 聚合后的特征表


In [1]:
# 大白话：这一格负责把原始CSV读进来，把 request_time 统一解析成时间，并准备 A/C/D 的兜底标记（不做复杂衍生）。

# 代码格 1/4：读取数据（逐行注释版）
# 目标：读入 CSV、解析 request_time、并对 A(关键列缺失) 做 -999 兜底准备
#
# ========================= 这套衍生“到底衍生了什么”（大白话说明）=========================
# 1) 一共衍生了多少个特征？（你要先知道“规模”和“长相”）
#    这套 creditos（信贷板块）本质是：
#    **按时间窗口筛选明细 → 再按不同分组维度（机构大类 / tipoCuenta / tipoCredito）聚合 → 输出数量/占比 + 指标统计。**
#
#    - 时间窗口：7 个（30/60/90/120/180/360/720 天）
#    - 分组维度一共 3 套：
#      * 机构大类：17 个（nombreOtorgante 映射；含新增：个征信机构=SIC）
#      * tipoCuenta：7 个（F/H/R/Q/A/E/P）
#      * tipoCredito：5 个（CC/PP/TC/LC/PB）
#
#    - 每个窗口里会输出这些“板块特征”：
#      [板块A：窗口内总量] 1 个：creditos_{N}d_total_cnt（近 N 天内一共有多少条 creditos 明细）
#
#      [板块B：机构大类结构] 17 类 × 2 个 = 34 个：
#        - cnt：该类条数
#        - ratio：该类条数 / total_cnt
#      [板块C：机构大类数值画像] 17 类 × 16 指标 × 5 统计 = 1360 个
#
#      [板块D：tipoCuenta 结构] 7 类 × 2 个 = 14 个
#      [板块E：tipoCuenta 数值画像] 7 类 × 16 指标 × 5 统计 = 560 个
#
#      [板块F：tipoCredito 结构] 5 类 × 2 个 = 10 个
#      [板块G：tipoCredito 数值画像] 5 类 × 16 指标 × 5 统计 = 400 个
#
#    - 所以：
#      * 每窗口特征数 = 1 + (34+1360) + (14+560) + (10+400) = 2379
#      * 总特征数 = 2379 × 7 = 16653
#
#    另外：输出表里会带上 request_time（原始截止时间），它不是衍生特征。
#
# 2) 每个“数值画像”到底在算什么？（mean/std/min/max/notNull占比）
#    这里的 16 个“指标”包括两类：
#    - 金额/余额/额度/次数/逾期：比如 montoPagar、saldoActual、limiteCredito、saldoVencido、numeroPagos、peorAtraso...
#    - 日期差（天）/状态：比如 days_since_open、days_since_last_payment、is_closed...
#
#    对每个指标我们输出 5 个统计：
#    - mean：平均值（典型看“水平”）
#    - std：标准差（典型看“波动”）
#    - min/max：最小/最大（典型看“极值”）
#    - notNull占比：这个指标在该分组里**有值的比例**（有值条数/该分组总条数，用来反映数据质量/可计算性）
#
# 3) 核心逻辑（窗口 / 分组 / 时间对比怎么做）
#    - 两个时间点是谁？
#      * 截止时间：request_time（每个 apply_id 自己的截止日）
#      * 窗口日期：fechaAperturaCuenta（开户日期，按你要求）
#
#    - 时间对比的计算怎么做？
#      * 先把时间字段统一解析成 datetime：pd.to_datetime(..., errors='coerce')
#      * 然后做相减：request_time - fechaAperturaCuenta -> 得到一个 Timedelta
#      * 再把 Timedelta 转成“天”：timedelta.dt.total_seconds() / 86400
#      * 得到 days_before_request_credit（可以带小数，原因是 request_time 可能带时分秒；只有年月日的日期会按 00:00:00 处理）
#
#    - 窗口筛选条件：
#        0 <= days_before_request_credit <= N
#      含义：只统计“在截止日前 N 天内开户”的账户记录。
#
#    - 分组维度（做三套同口径的分组）：
#      * apply_id + otorgante_group（机构大类 17 类）
#      * apply_id + tipo_cuenta（7 类）
#      * apply_id + tipo_credito（5 类）
#
# 4) 各种缺失/异常怎么处理？你最终会看到 0 / NaN / -999 三种表现
#    - A/C/D（你指定的硬规则）：
#      * 情况 A：缺关键列（apply_id/request_time/response_body）
#      * 情况 C：response_body 为空/非法 JSON
#      * 情况 D：JSON 合法但没有 creditos（或 creditos 不是 list）
#      => 命中时：该 apply_id 的**所有衍生特征统一 -999**。
#
#    - creditos 存在但为空列表 []：
#      * 这不算 D（结构是对的，只是没有账户）
#      * 你会看到：total_cnt=0、各类 cnt=0、ratio=0；均值/标准差/min/max 因为没数据会是 NaN；notNull占比会是 0。
#
#    - 日期解析失败（变 NaT）：
#      * 窗口日期 fechaAperturaCuenta 解析失败 -> days_before_request_credit 为空 -> 这条明细会被过滤掉（不进窗口统计）。
#      * 其它日期字段（例如 fechaUltimoPago/fechaCierreCuenta）解析失败 -> 对应的日期差指标为 NaN -> 统计时 notNull占比会变低。
#
#    - 类型字段缺失/不在关注范围：
#      * nombreOtorgante 不在字典/缺失 -> 归为“其他”（不会进入 17 类统计列，但仍可能计入 total_cnt）。
#      * tipoCuenta/tipoCredito 缺失或不在你指定的类别清单里 -> 不进入对应类别统计列，但仍可能计入 total_cnt。
#
#    - 数值字段解析失败（转 numeric 失败）：
#      * 会变成 NaN；统计时 mean/std/min/max 会基于“有值的那部分”计算；notNull占比会反映缺失程度。
# =============================================================================

# 大白话：导入模块：后面代码要用到它。
import json  # 用于解析 response_body 的 JSON 字符串
# 大白话：从模块里导入需要的对象：后面直接用名字调用。
from pathlib import Path  # 用于更安全地处理文件路径

# 大白话：导入模块：后面代码要用到它。
import numpy as np  # 用于 NaN、数值计算
# 大白话：导入模块：后面代码要用到它。
import pandas as pd  # 用于 DataFrame 处理与聚合

# 大白话：你现在要用筛选后的 pickle 数据做第二板块衍生，所以这里增加“优先读 pickle”的逻辑。
# 大白话：默认优先读取 cdc_pickle_pass_fpd7.pkl；如果你想回到 CSV，把 USE_PICKLE 改成 False。

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
USE_PICKLE = True  # True=优先读pickle；False=读CSV

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
INPUT_PICKLE = Path("cdc_pickle_pass_fpd7.pkl")  # 你筛选后的数据

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
INPUT_CSV = Path("衍生原始数据_331条.csv")  # 原始 CSV（兜底/回退用）

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
OUTPUT_DIR = Path("outputs")  # 输出目录（默认不写文件；仅在你打开 WRITE_OUTPUTS 时写）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # 确保 outputs 目录存在

# 大白话：读入数据（优先 pickle；否则读 CSV）。
if USE_PICKLE and INPUT_PICKLE.exists():
    df_raw = pd.read_pickle(INPUT_PICKLE)
    print("[INFO] loaded pickle:", INPUT_PICKLE)
else:
    df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    print("[INFO] loaded csv:", INPUT_CSV)

# 关键字段：apply_id / request_time / response_body
# 大白话：你这份 cdc_pickle_pass_fpd7.pkl 里没有 request_time，而是用 apply_time 表示“截止日期”。
# 大白话：为了不改变你之前的口径（仍然用 request_time 做窗口截止），这里做一个兼容：apply_time -> request_time。
if ("request_time" not in df_raw.columns) and ("apply_time" in df_raw.columns):
    df_raw["request_time"] = df_raw["apply_time"]
    print("[INFO] mapped apply_time -> request_time")

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
required_cols = ["apply_id", "request_time", "response_body"]
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
missing = [c for c in required_cols if c not in df_raw.columns]

# === 情况 A：关键字段缺失 -> 该 apply_id 的衍生特征统一 -999 ===
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
FATAL_MISSING_COLUMNS = len(missing) > 0

# 大白话：条件判断：满足条件才执行下面的代码块。
if FATAL_MISSING_COLUMNS:
# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
    print("[WARN] 命中情况A：关键字段缺失 -> 后续该申请的衍生特征将统一填 -999")
# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
    print("[WARN] 缺失字段:", missing)

    # 为了让后续流程还能跑（并输出 -999 特征），补齐缺失列
# 大白话：条件判断：满足条件才执行下面的代码块。
    if "apply_id" not in df_raw.columns:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        df_raw["apply_id"] = pd.Series(dtype="int64")
# 大白话：条件判断：满足条件才执行下面的代码块。
    if "request_time" not in df_raw.columns:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        df_raw["request_time"] = pd.NaT
# 大白话：条件判断：满足条件才执行下面的代码块。
    if "response_body" not in df_raw.columns:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        df_raw["response_body"] = None

# 解析 request_time（解析失败会变 NaT，后面会被当作异常样本处理并置 -999）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
df_raw["request_time"] = pd.to_datetime(df_raw["request_time"], errors="coerce")

# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
print("df_raw shape:", df_raw.shape)
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
df_raw.head(3)


[INFO] loaded pickle: cdc_pickle_pass_fpd7.pkl
[INFO] mapped apply_time -> request_time
df_raw shape: (12546, 12)


,apply_id,response_body,apply_time,approve_state,credit_limit_amount,use_amount,principal_amount_borrowed,fpd7,spd7,credit_apply_cnt,blind_lend,request_time
0,1065991091661283329,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 01:53:44.943,CYCLE_PASS,800.00000000,700.00000000,700.00000000,0,0,1,NaN,2025-11-25 01:53:44.943
1,1066560157648134145,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 06:29:46.980,CYCLE_PASS,500.00000000,500.00000000,500.00000000,0,0,1,1.0,2025-11-25 06:29:46.980
2,1066719243236777985,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 07:46:56.302,SINGLE_PASS,500.00000000,500.00000000,500.00000000,0,0,1,NaN,2025-11-25 07:46:56.302


In [ ]:
# 大白话：这一格负责把 response_body.creditos 解析并平铺成明细表，做机构归类、tipo字段标准化、计算窗口用的天数差，以及把日期/数值字段转成可统计的数值。

# 代码格 2/4：处理数据（逐行注释版）
# 目标：
# - 解析 response_body.creditos
# - 平铺成 creditos_df（apply_id 重复）
# - 机构归类（17 类，新增：个征信机构=SIC）
# - 计算窗口过滤用的“开户日期差”：request_time - fechaAperturaCuenta
# - 把你要的日期字段全部转成“天数差/可统计数值”

# ========== 机构归类字典（与你提供的一致）==========
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
OTORGANTE_GROUP_DICT = {
# 大白话：执行这一行代码，推进整个处理流程。
    "商店信息": ["TIENDA COMERCIAL", "TIENDA DE AUTOSERVICIO", "TIENDA DEPARTAMENTAL"],
# 大白话：执行这一行代码，推进整个处理流程。
    "大众金融协会": [
# 大白话：执行这一行代码，推进整个处理流程。
        "SOCIEDADES FINANCIERAS POPULARES",
# 大白话：执行这一行代码，推进整个处理流程。
        "SOCIEDAD FINANCIERA COMUNITARIA",
# 大白话：执行这一行代码，推进整个处理流程。
        "ARRENDADORAS FINANCIERAS",
# 大白话：执行这一行代码，推进整个处理流程。
        "UNION DE CREDITO",
# 大白话：执行这一行代码，推进整个处理流程。
    ],
# 大白话：执行这一行代码，推进整个处理流程。
    "多用途": ["SOCIEDAD FINANCIERA DE OBJETO MULTIPLE"],
# 大白话：执行这一行代码，推进整个处理流程。
    "非银行抵押": ["HIPOTECARIO NO BANCARIO"],
# 大白话：执行这一行代码，推进整个处理流程。
    "服务信息": ["SALUD Y SERVICIOS MEDICOS", "SERVICIO MEDICO", "SERVS. GRALES.", "SERVICIOS"],
# 大白话：执行这一行代码，推进整个处理流程。
    "付费电视": ["SERVICIO DE TELEVISION DE PAGA"],
# 大白话：执行这一行代码，推进整个处理流程。
    "个人贷款": ["COMPANIA DE PRESTAMO PERSONAL", "SOFOL PRESTAMO PERSONAL"],
# 大白话：执行这一行代码，推进整个处理流程。
    "基金和信托": ["FONDOS Y FIDEIC", "FONDOS Y FIDEICOMISOS", "FONDOS Y FIDEICO"],
# 大白话：执行这一行代码，推进整个处理流程。
    "建筑": ["MERCANCIA PARA LA CONSTRUCCION"],
# 大白话：执行这一行代码，推进整个处理流程。
    "金融公司": ["SOFOL EMPRESARIAL", "SOFOL AUTOMOTRIZ", "OTRAS FINANCIERA", "ARRENDADORAS NO FINANCIERAS"],
# 大白话：执行这一行代码，推进整个处理流程。
    "通讯": ["TELEFONIA LOCAL Y DE LARGA DISTANCIA", "TELEFONIA CELULAR", "COMUNICACIONES"],
# 大白话：执行这一行代码，推进整个处理流程。
    "销售": ["VENTA POR CATALOGO"],
# 大白话：执行这一行代码，推进整个处理流程。
    "小额贷款": ["MIC CREDITO PERS", "MICROFINANCIERA"],
# 大白话：执行这一行代码，推进整个处理流程。
    "银行": ["BANCOS", "BANCO"],
# 大白话：执行这一行代码，推进整个处理流程。
    "非银行": ["FINANCIERA"],
# 大白话：执行这一行代码，推进整个处理流程。
    "政府": ["GUBERNAMENTALES", "GOBIERNO", "HIPOTECAGOBIERNO"],
# 大白话：执行这一行代码，推进整个处理流程。
    "个征信机构": ["SIC"],
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
}

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
GROUPS_TO_USE = list(OTORGANTE_GROUP_DICT.keys())  # 17 个大类

# 机构大类中文名 -> 英文ID（用于特征列名，避免出现中文）
GROUP_NAME_TO_ID = {
    "商店信息": "shop",
    "大众金融协会": "mass_fin_assn",
    "多用途": "multi_purpose",
    "非银行抵押": "nonbank_mortgage",
    "服务信息": "service",
    "付费电视": "paytv",
    "个人贷款": "personal_loan",
    "基金和信托": "fund_trust",
    "建筑": "construction",
    "金融公司": "finance_company",
    "通讯": "telecom",
    "销售": "catalog_sale",
    "小额贷款": "micro_loan",
    "银行": "bank",
    "非银行": "nonbank",
    "政府": "government",
    "个征信机构": "sic",
}
# 英文ID -> 中文名（用于特征字典描述）
GROUP_ID_TO_NAME = {v: k for k, v in GROUP_NAME_TO_ID.items()}

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
VALUE_TO_GROUP = {raw: g for g, vals in OTORGANTE_GROUP_DICT.items() for raw in vals}  # 反向映射


# 大白话：定义一个函数：把一段逻辑封装起来，后面反复调用。
def map_otorgante_group(nombre_otorgante: str) -> str:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    """把 nombreOtorgante 映射到 17 类（新增：个征信机构=SIC），不在字典/缺失 -> 其他。"""
# 大白话：条件判断：满足条件才执行下面的代码块。
    if pd.isna(nombre_otorgante):
# 大白话：返回结果：把函数的输出交回给调用方。
        return "其他"
# 大白话：返回结果：把函数的输出交回给调用方。
    return VALUE_TO_GROUP.get(str(nombre_otorgante), "其他")


# ========== tipoCuenta / tipoCredito 的统计口径（你指定的类别）==========
# tipoCuenta：F 固定付款；H 抵押贷款；R 循环信用；Q 无担保信用；A 授信或预支信用；E 分期付款信用；P 质押信用
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
TIPO_CUENTA_TO_USE = ["F", "H", "R", "Q", "A", "E", "P"]
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
TIPO_CUENTA_TO_ID = {c: c.lower() for c in TIPO_CUENTA_TO_USE}  # 用于列名（例如 F->f）

# tipoCredito：CC 消费信贷；PP 个人贷款；TC 信用卡；LC 信用额度；PB 个人银行贷款
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
TIPO_CREDITO_TO_USE = ["CC", "PP", "TC", "LC", "PB"]
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
TIPO_CREDITO_TO_ID = {c: c.lower() for c in TIPO_CREDITO_TO_USE}  # 用于列名（例如 CC->cc）


# ========== 解析并平铺 creditos ==========
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
rows = []

# apply_status：记录每个 apply_id 是否命中 A/C/D（用于最后把该申请的衍生特征统一置为 -999）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
apply_status = {aid: "ok" for aid in df_raw["apply_id"].tolist()}

# 如果命中情况 A，则所有 apply_id 标记为 A
# 大白话：条件判断：满足条件才执行下面的代码块。
if "FATAL_MISSING_COLUMNS" in globals() and FATAL_MISSING_COLUMNS:
# 大白话：开始循环：对一组元素逐个处理。
    for aid in apply_status:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        apply_status[aid] = "A_missing_columns"

# 大白话：开始循环：对一组元素逐个处理。
for apply_id, request_time, response_body in df_raw[["apply_id", "request_time", "response_body"]].itertuples(
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    index=False, name=None
# 大白话：执行这一行代码，推进整个处理流程。
):
    # 情况 C：response_body 为空/非法 JSON
# 大白话：条件判断：满足条件才执行下面的代码块。
    if response_body is None or (isinstance(response_body, float) and pd.isna(response_body)) or str(response_body).strip() == "":
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        apply_status[apply_id] = "C_bad_json"
# 大白话：执行这一行代码，推进整个处理流程。
        continue

# 大白话：异常保护：尝试执行，失败就交给 except 处理。
    try:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        obj = json.loads(response_body) if isinstance(response_body, str) else response_body
# 大白话：捕获异常：上一段 try 出错时走这里，避免程序中断。
    except Exception:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        apply_status[apply_id] = "C_bad_json"
# 大白话：执行这一行代码，推进整个处理流程。
        continue

    # 情况 D：没有 creditos 块（key 不存在）
# 大白话：条件判断：满足条件才执行下面的代码块。
    if not isinstance(obj, dict) or ("creditos" not in obj):
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        apply_status[apply_id] = "D_no_creditos_key"
# 大白话：执行这一行代码，推进整个处理流程。
        continue

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    creditos = obj.get("creditos")

    # 情况 D（扩展）：creditos 存在但不是 list
# 大白话：条件判断：满足条件才执行下面的代码块。
    if not isinstance(creditos, list):
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        apply_status[apply_id] = "D_no_creditos_key"
# 大白话：执行这一行代码，推进整个处理流程。
        continue

    # 正常情况：creditos 是 list（可以为空 list，这不属于 D，只是没有信贷账户记录）
# 大白话：开始循环：对一组元素逐个处理。
    for it in creditos:
# 大白话：条件判断：满足条件才执行下面的代码块。
        if not isinstance(it, dict):
# 大白话：执行这一行代码，推进整个处理流程。
            continue

        # 平铺字段：按你要求的字段名取出
# 大白话：执行这一行代码，推进整个处理流程。
        rows.append(
# 大白话：执行这一行代码，推进整个处理流程。
            {
# 大白话：执行这一行代码，推进整个处理流程。
                "apply_id": apply_id,
# 大白话：执行这一行代码，推进整个处理流程。
                "request_time": request_time,
                # 机构类别
# 大白话：执行这一行代码，推进整个处理流程。
                "nombreOtorgante": it.get("nombreOtorgante"),
                # 金额/额度/余额/次数等
# 大白话：执行这一行代码，推进整个处理流程。
                "montoPagar": it.get("montoPagar"),
# 大白话：执行这一行代码，推进整个处理流程。
                "saldoActual": it.get("saldoActual"),
# 大白话：执行这一行代码，推进整个处理流程。
                "limiteCredito": it.get("limiteCredito"),
# 大白话：执行这一行代码，推进整个处理流程。
                "saldoVencido": it.get("saldoVencido"),
# 大白话：执行这一行代码，推进整个处理流程。
                "valorActivoValuacion": it.get("valorActivoValuacion"),
# 大白话：执行这一行代码，推进整个处理流程。
                "creditoMaximo": it.get("creditoMaximo"),
# 大白话：执行这一行代码，推进整个处理流程。
                "numeroPagos": it.get("numeroPagos"),
# 大白话：执行这一行代码，推进整个处理流程。
                "peorAtraso": it.get("peorAtraso"),
# 大白话：执行这一行代码，推进整个处理流程。
                "saldoVencidoPeorAtraso": it.get("saldoVencidoPeorAtraso"),
                # 日期字段
# 大白话：执行这一行代码，推进整个处理流程。
                "fechaUltimaCompra": it.get("fechaUltimaCompra"),
# 大白话：执行这一行代码，推进整个处理流程。
                "fechaAperturaCuenta": it.get("fechaAperturaCuenta"),
# 大白话：执行这一行代码，推进整个处理流程。
                "fechaCierreCuenta": it.get("fechaCierreCuenta"),
# 大白话：执行这一行代码，推进整个处理流程。
                "fechaUltimoPago": it.get("fechaUltimoPago"),
# 大白话：执行这一行代码，推进整个处理流程。
                "fechaPeorAtraso": it.get("fechaPeorAtraso"),
                # 类型字段
# 大白话：执行这一行代码，推进整个处理流程。
                "tipoCuenta": it.get("tipoCuenta"),
# 大白话：执行这一行代码，推进整个处理流程。
                "tipoCredito": it.get("tipoCredito"),
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
            }
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
        )

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df = pd.DataFrame(rows)

# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
print("creditos_df shape (flat):", creditos_df.shape)

# ========== 类型转换（日期/数值）==========
# 日期统一转 datetime
# 大白话：开始循环：对一组元素逐个处理。
for c in [
# 大白话：执行这一行代码，推进整个处理流程。
    "fechaUltimaCompra",
# 大白话：执行这一行代码，推进整个处理流程。
    "fechaAperturaCuenta",
# 大白话：执行这一行代码，推进整个处理流程。
    "fechaCierreCuenta",
# 大白话：执行这一行代码，推进整个处理流程。
    "fechaUltimoPago",
# 大白话：执行这一行代码，推进整个处理流程。
    "fechaPeorAtraso",
# 大白话：执行这一行代码，推进整个处理流程。
]:
# 大白话：条件判断：满足条件才执行下面的代码块。
    if c in creditos_df.columns:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        creditos_df[c] = pd.to_datetime(creditos_df[c], errors="coerce")

# 数值字段统一转 numeric（无法解析的变 NaN）
# 大白话：开始循环：对一组元素逐个处理。
for c in [
# 大白话：执行这一行代码，推进整个处理流程。
    "montoPagar",
# 大白话：执行这一行代码，推进整个处理流程。
    "saldoActual",
# 大白话：执行这一行代码，推进整个处理流程。
    "limiteCredito",
# 大白话：执行这一行代码，推进整个处理流程。
    "saldoVencido",
# 大白话：执行这一行代码，推进整个处理流程。
    "valorActivoValuacion",
# 大白话：执行这一行代码，推进整个处理流程。
    "creditoMaximo",
# 大白话：执行这一行代码，推进整个处理流程。
    "numeroPagos",
# 大白话：执行这一行代码，推进整个处理流程。
    "peorAtraso",
# 大白话：执行这一行代码，推进整个处理流程。
    "saldoVencidoPeorAtraso",
# 大白话：执行这一行代码，推进整个处理流程。
]:
# 大白话：条件判断：满足条件才执行下面的代码块。
    if c in creditos_df.columns:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        creditos_df[c] = pd.to_numeric(creditos_df[c], errors="coerce")

# ========== 机构归类 + tipo字段标准化 + 窗口过滤用天数差 ==========
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["otorgante_group"] = creditos_df["nombreOtorgante"].apply(map_otorgante_group)

# 标准化 tipoCuenta / tipoCredito：统一转大写、去空格；缺失置空字符串（便于 isin 判断）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["tipo_cuenta"] = (
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
    creditos_df["tipoCuenta"].fillna("").astype(str).str.strip().str.upper()
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["tipo_credito"] = (
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
    creditos_df["tipoCredito"].fillna("").astype(str).str.strip().str.upper()
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# ========== dtype 瘦身（路线A）：让大数据更省内存、更稳定 ==========
# 大白话：把分组字段转为 category，可以显著减少内存占用（尤其是 1w+ 样本、百万级明细行时）。
creditos_df["otorgante_group"] = creditos_df["otorgante_group"].astype("category")
creditos_df["tipo_cuenta"] = creditos_df["tipo_cuenta"].astype("category")
creditos_df["tipo_credito"] = creditos_df["tipo_credito"].astype("category")

# 大白话：把数值类列尽量压到 float32（你最终只保留2位小数，float32 足够）。
for _c in [
    "montoPagar",
    "saldoActual",
    "limiteCredito",
    "saldoVencido",
    "valorActivoValuacion",
    "creditoMaximo",
    "numeroPagos",
    "peorAtraso",
    "saldoVencidoPeorAtraso",
    "days_since_last_payment",
    "days_since_open",
    "days_since_close",
    "is_closed",
    "days_pay_since_open",
    "days_worst_arrears_since_open",
    "days_since_worst_arrears_date",
]:
    if _c in creditos_df.columns:
        creditos_df[_c] = pd.to_numeric(creditos_df[_c], errors="coerce").astype("float32")

# 开户日期差：request_time - fechaAperturaCuenta（天），用于窗口过滤
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["days_before_request_credit"] = (
# 大白话：执行这一行代码，推进整个处理流程。
    (creditos_df["request_time"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# 大白话：为了做“17类命中率”诊断（只看 nombreOtorgante 是否出现，不受开户日期缺失影响），这里先保留一份“不过滤”的全量明细。
creditos_df_all = creditos_df.copy()

# 过滤：未来开户（负数）与 fechaAperturaCuenta 解析失败
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df = creditos_df[
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    creditos_df["days_before_request_credit"].notna() & (creditos_df["days_before_request_credit"] >= 0)
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
].copy()

# ========== 防穿越：把“晚于 request_time 的日期”统一视为无效（置为 NaT）==========
# 大白话：我们只允许使用“截止日期 request_time 及之前”的信息；任何日期字段如果 > request_time，都属于未来信息，不参与衍生。
# 大白话：说明：这里只列“第二板块实际平铺且会参与日期差计算”的日期字段；不会混入其它 Notebook（如 BOSS）的字段。
# 大白话：执行这一行代码，推进整个处理流程。
for _c in [
    "fechaCierreCuenta",
    "fechaUltimoPago",
    "fechaPeorAtraso",
    "fechaUltimaCompra",
    "fechaActualizacion",
    "ultimaFechaSaldoCero",
]:
    # 大白话：这个 notebook 只会平铺你指定的字段；如果某个日期列根本不存在，就跳过（避免 KeyError）。
    if _c not in creditos_df.columns:
        continue
    # 大白话：只要该日期字段存在、且比 request_time 晚，就置为 NaT。
    creditos_df.loc[
        creditos_df[_c].notna()
        & creditos_df["request_time"].notna()
        & (creditos_df[_c] > creditos_df["request_time"]),
        _c,
    ] = pd.NaT

# ========== 你要的“计算数据”转换为可统计的数值（主要是各种日期差）==========
# 距付款日期天数：request_time - fechaUltimoPago
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["days_since_last_payment"] = (
# 大白话：执行这一行代码，推进整个处理流程。
    (creditos_df["request_time"] - creditos_df["fechaUltimoPago"]).dt.total_seconds() / 86400
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# 距开户日期天数：request_time - fechaAperturaCuenta
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["days_since_open"] = (
# 大白话：执行这一行代码，推进整个处理流程。
    (creditos_df["request_time"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# 距账户关闭天数：request_time - fechaCierreCuenta（未关闭则 NaN）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["days_since_close"] = (
# 大白话：执行这一行代码，推进整个处理流程。
    (creditos_df["request_time"] - creditos_df["fechaCierreCuenta"]).dt.total_seconds() / 86400
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# 账户关闭或者未关闭：is_closed（关闭=1，未关闭=0）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["is_closed"] = creditos_df["fechaCierreCuenta"].notna().astype(int)

# 支付距开户天数：fechaUltimoPago - fechaAperturaCuenta
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["days_pay_since_open"] = (
# 大白话：执行这一行代码，推进整个处理流程。
    (creditos_df["fechaUltimoPago"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# 大白话：数据一致性兜底（非常关键）：如果出现“最后还款日期早于开户日期”，则 days_pay_since_open 会是负数。
# 大白话：这种情况在业务含义上不合理（同一账户不应在开户前还款），我们将其视为脏数据并置为 NaN，避免把负值带进统计导致特征异常。
creditos_df.loc[creditos_df["days_pay_since_open"] < 0, "days_pay_since_open"] = np.nan

# 最严重逾期距开户日期：fechaPeorAtraso - fechaAperturaCuenta
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["days_worst_arrears_since_open"] = (
# 大白话：执行这一行代码，推进整个处理流程。
    (creditos_df["fechaPeorAtraso"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# 大白话：数据一致性兜底：如果“最严重逾期日期早于开户日期”，则该差值为负。
# 大白话：同理视为脏数据并置 NaN，避免统计口径被异常负值污染。
creditos_df.loc[creditos_df["days_worst_arrears_since_open"] < 0, "days_worst_arrears_since_open"] = np.nan

# 当前日期距离最严重逾期日期：request_time - fechaPeorAtraso
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_df["days_since_worst_arrears_date"] = (
# 大白话：执行这一行代码，推进整个处理流程。
    (creditos_df["request_time"] - creditos_df["fechaPeorAtraso"]).dt.total_seconds() / 86400
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
print("creditos_df shape (after cleaning):", creditos_df.shape)
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
creditos_df.head(5)


creditos_df shape (flat): (697815, 19)


creditos_df shape (after cleaning): (697815, 30)


,apply_id,request_time,nombreOtorgante,montoPagar,saldoActual,limiteCredito,saldoVencido,valorActivoValuacion,creditoMaximo,numeroPagos,...,tipo_cuenta,tipo_credito,days_before_request_credit,days_since_last_payment,days_since_open,days_since_close,is_closed,days_pay_since_open,days_worst_arrears_since_open,days_since_worst_arrears_date
0,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,0.0,0.0,0.0,0.0,NaN,7774.0,26.0,...,F,PP,1034.078992,803.078992,1034.078992,803.078992,1,231.0,70.0,964.078992
1,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,0.0,0.0,0.0,0.0,8500.0,8500.0,100.0,...,F,PP,701.078992,56.078992,701.078992,56.078992,1,645.0,437.0,264.078992
2,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,84.0,7560.0,0.0,0.0,8400.0,8400.0,100.0,...,F,PP,77.078992,12.078992,77.078992,NaN,0,65.0,44.0,33.078992
3,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,0.0,0.0,0.0,0.0,2000.0,4004.0,52.0,...,F,PP,1017.078992,734.078992,1017.078992,734.078992,1,283.0,56.0,961.078992
4,1065991091661283329,2025-11-25 01:53:44.943,SOCIEDAD FINANCIERA DE OBJETO MULTIPLE,0.0,0.0,20000.0,0.0,NaN,600.0,1.0,...,F,PP,294.078992,278.078992,294.078992,278.078992,1,16.0,NaN,NaN


In [3]:
# 大白话：这一格把‘明细表’按窗口聚合成‘特征表’：对机构大类 / tipoCuenta / tipoCredito 分组输出 cnt/ratio，并对各指标算 mean/std/min/max/notNull占比，同时对 A/C/D 的申请统一返回 -999。

# 代码格 3/4：衍生函数逻辑（逐行注释版）
# 目标：按窗口（30/60/90/120/180/360/720天）+ 按机构 17 类 聚合统计你要求的所有指标（含新增：个征信机构=SIC）


# 大白话：定义一个函数：把一段逻辑封装起来，后面反复调用。
def _agg_numeric_stats(df: pd.DataFrame, group_cols: list[str], value_col: str) -> pd.DataFrame:
    """对某个数值列做 5 个统计：mean/std/min/max/notnull_ratio。

    - mean/std/min/max：常规聚合
    - notnull_ratio：非空占比 = 非空数量 / 总数量
# 大白话：执行这一行代码，推进整个处理流程。
    """

    # 总数量（分母）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    # 大白话：groupby 的 observed 默认值将来会变，为了避免 FutureWarning 且保持当前行为，这里显式写 observed=False。
    total_cnt = df.groupby(group_cols, observed=False).size().rename("__cnt").to_frame()

    # 非空数量
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    notnull_cnt = df.groupby(group_cols, observed=False)[value_col].count().rename("__notnull").to_frame()

    # 统计聚合
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    stats = df.groupby(group_cols, observed=False)[value_col].agg(["mean", "std", "min", "max"])  # std 默认 ddof=1

    # 合并并计算占比
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    out = stats.join(total_cnt).join(notnull_cnt)
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    out["notnull_ratio"] = (out["__notnull"] / out["__cnt"]).replace([np.inf, -np.inf], np.nan)

    # 清理中间列
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    out = out.drop(columns=["__cnt", "__notnull"])

# 大白话：返回结果：把函数的输出交回给调用方。
    return out


# 大白话：定义一个函数：把一段逻辑封装起来，后面反复调用。
def derive_creditos_features(
# 大白话：执行这一行代码，推进整个处理流程。
    df_raw: pd.DataFrame,
# 大白话：执行这一行代码，推进整个处理流程。
    creditos_df: pd.DataFrame,
# 大白话：执行这一行代码，推进整个处理流程。
    window_days_list: list[int],
# 大白话：执行这一行代码，推进整个处理流程。
    groups_to_use: list[str],
# 大白话：执行这一行代码，推进整个处理流程。
    tipo_cuenta_to_use: list[str],
# 大白话：执行这一行代码，推进整个处理流程。
    tipo_cuenta_to_id: dict[str, str],
# 大白话：执行这一行代码，推进整个处理流程。
    tipo_credito_to_use: list[str],
# 大白话：执行这一行代码，推进整个处理流程。
    tipo_credito_to_id: dict[str, str],
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    apply_status: dict | None = None,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    sentinel_value: float = -999,
# 大白话：执行这一行代码，推进整个处理流程。
) -> pd.DataFrame:
    """对 creditos 板块做特征衍生。

    每个窗口 N 天内：
    - 总数量 total_cnt
    - 机构大类：每类别数量 cnt / ratio + 指标统计（mean/std/min/max/notnull_ratio）
    - tipoCuenta：每类别数量 cnt / ratio + 指标统计（mean/std/min/max/notnull_ratio）
    - tipoCredito：每类别数量 cnt / ratio + 指标统计（mean/std/min/max/notnull_ratio）

    注：窗口过滤使用 days_before_request_credit（request_time - fechaAperturaCuenta）。
# 大白话：执行这一行代码，推进整个处理流程。
    """

    # 基表：每个 apply_id 一行
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    df_base = (
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
        df_raw[["apply_id", "request_time"]]
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
        .drop_duplicates("apply_id")
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
        .set_index("apply_id")
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
        .sort_index()
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
    )

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    features = df_base.copy()

    # 要做统计的数值列（你要求的“计算数据”统一用数值表达）
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    METRIC_COLS = {
        # 金额/余额/额度/次数
# 大白话：执行这一行代码，推进整个处理流程。
        "montoPagar": "monto_pagar",  # 当期应还金额
# 大白话：执行这一行代码，推进整个处理流程。
        "saldoActual": "saldo_actual",  # 当前总待还金额
# 大白话：执行这一行代码，推进整个处理流程。
        "limiteCredito": "limite_credito",  # 信用额度
# 大白话：执行这一行代码，推进整个处理流程。
        "saldoVencido": "saldo_vencido",  # 逾期余额（逾期金额）
# 大白话：执行这一行代码，推进整个处理流程。
        "valorActivoValuacion": "valor_activo_valuacion",  # 资产估值金额
# 大白话：执行这一行代码，推进整个处理流程。
        "creditoMaximo": "credito_maximo",  # 历史最高授信额
# 大白话：执行这一行代码，推进整个处理流程。
        "numeroPagos": "numero_pagos",  # 还款次数
# 大白话：执行这一行代码，推进整个处理流程。
        "peorAtraso": "peor_atraso",  # 最严重逾期期数
# 大白话：执行这一行代码，推进整个处理流程。
        "saldoVencidoPeorAtraso": "saldo_vencido_peor_atraso",  # 最严重逾期时的逾期余额
        # 日期差（天）
# 大白话：执行这一行代码，推进整个处理流程。
        "days_since_last_payment": "days_since_last_payment",  # 距付款日期天数（距最后还款日期天数）
# 大白话：执行这一行代码，推进整个处理流程。
        "days_since_open": "days_since_open",  # 距开户日期天数
# 大白话：执行这一行代码，推进整个处理流程。
        "days_since_close": "days_since_close",  # 距账户关闭天数
# 大白话：执行这一行代码，推进整个处理流程。
        "is_closed": "is_closed",  # 账户关闭或者未关闭（0/1）
# 大白话：执行这一行代码，推进整个处理流程。
        "days_pay_since_open": "days_pay_since_open",  # 支付距开户天数（最后还款日-开户日）
# 大白话：执行这一行代码，推进整个处理流程。
        "days_worst_arrears_since_open": "days_worst_arrears_since_open",  # 最严重逾期距开户日期
# 大白话：执行这一行代码，推进整个处理流程。
        "days_since_worst_arrears_date": "days_since_worst_arrears_date",  # 当前日期距离最严重逾期日期
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
    }

    # 每个窗口生成特征并拼到总 features（路线A：用 dict 收集 + concat，避免 DataFrame fragmentation）
    # 大白话：可选：把每个窗口的结果落盘（parquet）以便你调试/复用；默认 False（不额外生成文件）。
    from pathlib import Path
    WRITE_INTERMEDIATE_PARQUET = False
    INTERMEDIATE_DIR = Path("outputs/_tmp_block2_windows")
    if WRITE_INTERMEDIATE_PARQUET:
        INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

# 大白话：开始循环：对一组元素逐个处理。
    for w in window_days_list:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        sub = creditos_df[
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
            (creditos_df["days_before_request_credit"] >= 0)
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
            & (creditos_df["days_before_request_credit"] <= w)
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
        ].copy()

        # 大白话：col_dict 用来收集该窗口的所有特征列（避免对 DataFrame 逐列 insert）。
        col_dict: dict[str, pd.Series] = {}

        # 总数量：窗口内所有 creditos 记录数量
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        total_cnt = sub.groupby("apply_id", observed=False).size().reindex(df_base.index).fillna(0).astype("int32")
        col_dict[f"creditos_{w}d_total_cnt"] = total_cnt

        # 大白话：ratio 的分母（总数为0时记 NaN，后面再 fill 0）。
        denom = total_cnt.replace(0, np.nan)

        # ==================== 机构大类（17类）====================
        sub_cat = sub[sub["otorgante_group"].isin(groups_to_use)].copy()
        cnt = (
            sub_cat.groupby(["apply_id", "otorgante_group"], observed=False).size().unstack().reindex(index=df_base.index, columns=groups_to_use)
        )
        cnt = cnt.fillna(0).astype("int32")
        ratio = cnt.div(denom, axis=0).fillna(0.0).astype("float32")

        for g in groups_to_use:
            gid = GROUP_NAME_TO_ID.get(g, str(g))
            col_dict[f"creditos_{w}d_{gid}_cnt"] = cnt[g]
            col_dict[f"creditos_{w}d_{gid}_ratio"] = ratio[g]

        # 大白话：机构画像的 16 个指标统计（每指标 5 统计）。
        for raw_col, col_id in METRIC_COLS.items():
            if raw_col not in sub_cat.columns:
                continue
            stats = _agg_numeric_stats(sub_cat, group_cols=["apply_id", "otorgante_group"], value_col=raw_col)
            stats.columns = [f"{col_id}__{c}" for c in stats.columns]

            # 大白话：关键修复——unstack 后的列是 MultiIndex（第1层=统计名，第2层=类别）。
            # 大白话：如果直接 columns=groups_to_use，会把 MultiIndex 列“打平丢掉”，导致所有 mean/std/min/max 变 NaN。
            wide = stats.unstack("otorgante_group").reindex(index=df_base.index)
            wide = wide.reindex(columns=pd.MultiIndex.from_product([stats.columns, groups_to_use]))

            for group_name in groups_to_use:
                gid = GROUP_NAME_TO_ID.get(group_name, str(group_name))
                for stat_name in ["mean", "std", "min", "max", "notnull_ratio"]:
                    key = f"{col_id}__{stat_name}"
                    if (key, group_name) in wide.columns:
                        col_dict[f"creditos_{w}d_{gid}_{col_id}_{stat_name}"] = wide[(key, group_name)].astype("float32")
                    else:
                        col_dict[f"creditos_{w}d_{gid}_{col_id}_{stat_name}"] = pd.Series(np.nan, index=df_base.index, dtype="float32")

        # ==================== 通用：按某个类别字段生成 cnt/ratio + 16指标统计 ====================
        def _add_category_to_dict(*, sub_df: pd.DataFrame, cat_col: str, cat_values: list[str], cat_to_id: dict[str, str], prefix: str) -> None:
            sub_cat_local = sub_df[sub_df[cat_col].isin(cat_values)].copy()

            cnt_local = (
                sub_cat_local.groupby(["apply_id", cat_col], observed=False).size().unstack().reindex(index=df_base.index, columns=cat_values)
            )
            cnt_local = cnt_local.fillna(0).astype("int32")
            ratio_local = cnt_local.div(denom, axis=0).fillna(0.0).astype("float32")

            for v in cat_values:
                vid = cat_to_id.get(v, str(v).lower())
                col_dict[f"creditos_{w}d_{prefix}_{vid}_cnt"] = cnt_local[v]
                col_dict[f"creditos_{w}d_{prefix}_{vid}_ratio"] = ratio_local[v]

            for raw_col, col_id in METRIC_COLS.items():
                if raw_col not in sub_cat_local.columns:
                    continue
                stats_local = _agg_numeric_stats(sub_cat_local, group_cols=["apply_id", cat_col], value_col=raw_col)
                stats_local.columns = [f"{col_id}__{c}" for c in stats_local.columns]

                # 大白话：同上，unstack 后列是 MultiIndex，需要按 (stat, category) 的笛卡尔积补齐列，避免把统计值整体丢掉。
                wide_local = stats_local.unstack(cat_col).reindex(index=df_base.index)
                wide_local = wide_local.reindex(columns=pd.MultiIndex.from_product([stats_local.columns, cat_values]))

                for v in cat_values:
                    vid = cat_to_id.get(v, str(v).lower())
                    for stat_name in ["mean", "std", "min", "max", "notnull_ratio"]:
                        key = f"{col_id}__{stat_name}"
                        if (key, v) in wide_local.columns:
                            col_dict[f"creditos_{w}d_{prefix}_{vid}_{col_id}_{stat_name}"] = wide_local[(key, v)].astype("float32")
                        else:
                            col_dict[f"creditos_{w}d_{prefix}_{vid}_{col_id}_{stat_name}"] = pd.Series(np.nan, index=df_base.index, dtype="float32")

        # === tipoCuenta（7类）===
        _add_category_to_dict(sub_df=sub, cat_col="tipo_cuenta", cat_values=tipo_cuenta_to_use, cat_to_id=tipo_cuenta_to_id, prefix="tipo_cuenta")

        # === tipoCredito（5类）===
        _add_category_to_dict(sub_df=sub, cat_col="tipo_credito", cat_values=tipo_credito_to_use, cat_to_id=tipo_credito_to_id, prefix="tipo_credito")

        # 大白话：把 notnull_ratio 的 NaN 统一补 0（无样本或无有效值时更符合直觉）。
        window_df = pd.DataFrame(col_dict, index=df_base.index)
        notnull_cols = [c for c in window_df.columns if c.endswith("_notnull_ratio")]
        if len(notnull_cols) > 0:
            window_df[notnull_cols] = window_df[notnull_cols].fillna(0.0).astype("float32")

        # 大白话：可选落盘：每个窗口单独保存一个 parquet（默认关闭）。
        if WRITE_INTERMEDIATE_PARQUET:
            window_df.reset_index().to_parquet(INTERMEDIATE_DIR / f"block2_window_{w}d.parquet", index=False)

        # 大白话：用 concat 把该窗口结果拼到 features（只拼 7 次，不会碎片化）。
        features = pd.concat([features, window_df], axis=1)

        # 大白话：释放临时变量，减小内存峰值。
        del window_df, col_dict

    # === A/C/D -> -999（只覆盖衍生特征列，不覆盖 request_time）===
# 大白话：条件判断：满足条件才执行下面的代码块。
    if apply_status is not None:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
        bad_ids = [
# 大白话：执行这一行代码，推进整个处理流程。
            aid
# 大白话：开始循环：对一组元素逐个处理。
            for aid, st in apply_status.items()
# 大白话：条件判断：满足条件才执行下面的代码块。
            if st in {"A_missing_columns", "C_bad_json", "D_no_creditos_key"}
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
        ]
# 大白话：条件判断：满足条件才执行下面的代码块。
        if len(bad_ids) > 0:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
            feature_cols = [c for c in features.columns if c != "request_time"]
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
            features.loc[bad_ids, feature_cols] = sentinel_value

# 大白话：返回结果：把函数的输出交回给调用方。
    return features


In [ ]:
# 大白话：这一格调用上一格的衍生函数真正生成 features，并用 WRITE_OUTPUTS 开关决定是否把两份CSV写到 outputs（默认不写，等你确认）。

# 代码格 4/4：生成结果（逐行注释版）
# 目标：计算 features 与 creditos_df；默认不落盘 outputs（你确认后再输出两份 CSV）

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
WINDOW_DAYS = [30, 60, 90, 120, 180, 360, 720]  # 你的时间窗口

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
features = derive_creditos_features(
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    df_raw=df_raw,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    creditos_df=creditos_df,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    window_days_list=WINDOW_DAYS,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    groups_to_use=GROUPS_TO_USE,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    tipo_cuenta_to_use=TIPO_CUENTA_TO_USE,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    tipo_cuenta_to_id=TIPO_CUENTA_TO_ID,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    tipo_credito_to_use=TIPO_CREDITO_TO_USE,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    tipo_credito_to_id=TIPO_CREDITO_TO_ID,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    apply_status=apply_status,
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    sentinel_value=-999,
# 大白话：这一行是上一段多行表达式/参数列表的收尾。
)

# === 删除与第三板块重复的“窗口内总记录数”特征列（只删第二板块）===
# 大白话：第二板块与第三板块都在 creditos 上做了 "total_cnt"，为了避免你后面横向合并时列名冲突，这里把第二板块的这 7 列删掉。
# 大白话：重复列是：creditos_{w}d_total_cnt（w ∈ WINDOW_DAYS）。
_dup_total_cnt_cols = [f"creditos_{w}d_total_cnt" for w in WINDOW_DAYS]
features = features.drop(columns=[c for c in _dup_total_cnt_cols if c in features.columns])

# === 输出开关（默认不输出，等你确认后再改成 True）===
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
WRITE_OUTPUTS = True

# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
creditos_out_path = OUTPUT_DIR / "creditos_flat.csv"  # 平铺明细
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
features_out_path = OUTPUT_DIR / "features_creditos.csv"  # 特征表

# 大白话：条件判断：满足条件才执行下面的代码块。
if WRITE_OUTPUTS:
# 大白话：把右边计算/取到的值，保存到左边变量/列里，供后面继续用。
    creditos_df.to_csv(creditos_out_path, index=False, encoding="utf-8-sig")

    # 只对“衍生特征列”加前缀/后缀（request_time 不加），并写出到 CSV
    _rename_map_for_write = {c: f"cdc_{c}_v2" for c in features.columns if c != "request_time"}

    _features_to_write = features.rename(columns=_rename_map_for_write).reset_index()  # apply_id 变回普通列，便于写 CSV
    _round_cols = [c for c in _features_to_write.columns if c not in {"apply_id", "request_time"}]  # 只对衍生特征列做保留2位小数
    _features_to_write[_round_cols] = _features_to_write[_round_cols].apply(pd.to_numeric, errors="coerce").round(2)  # 统一保留小数点后2位

    _features_to_write.to_csv(
        features_out_path,
        index=False,
        encoding="utf-8-sig",
    )

# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
    print("written:")
# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
    print("-", creditos_out_path)
# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
    print("-", features_out_path)
# 大白话：否则分支：上面条件都不满足时走这里。
else:
# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
    print("[INFO] 当前 WRITE_OUTPUTS=True：已计算完成，但未写出 outputs/*.csv（等你确认后再输出）")

# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
print("creditos_flat rows:", len(creditos_df))
# 大白话：打印提示信息：方便你在 Notebook 里快速核对运行结果。
print("features shape:", features.shape)

# 大白话：自检：如果你看到 request_time 是 NaT，通常是你没有从第1格开始重跑导致变量还是旧状态。
# 大白话：这里直接打印 request_time 的 NaT 比例和前几行，方便你一眼确认。
print("features.request_time NaT rate:", float(features["request_time"].isna().mean()))
print("features.request_time head:", features["request_time"].head(3).tolist())

# === 特征数据字典（不落盘，仅用于核对/拼接）===
# 大白话：这里生成一个“特征列名 -> 含义”的映射表 feature_dict，且列格式与另外两板块一致，方便直接上下拼接。

# 大白话：导入 re，用正则解析列名结构。
import re


# 大白话：把第二板块的特征列名解析成统一的数据字典格式。
def build_feature_dict_block2(cols: list[str]) -> pd.DataFrame:
    # 大白话：用 rows 收集每一行“特征定义”。
    rows = []

    # 大白话：逐个遍历特征列名。
    for col in cols:
        # 大白话：request_time 是原始字段，不算衍生特征，所以跳过。
        if col == "request_time":
            continue

        # 大白话：窗口总量特征：creditos_{w}d_total_cnt。
        m = re.match(r"^creditos_(\d+)d_total_cnt$", col)
        if m:
            w = int(m.group(1))
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "total",
                    "group_code": "ALL",
                    "metric": "count",
                    "stat": "cnt",
                    "description": f"第二板块：近{w}天内 creditos 总记录数",
                }
            )
            continue

        # 大白话：tipoCuenta 计数/占比。
        m = re.match(r"^creditos_(\d+)d_tipo_cuenta_([a-z]+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            k = m.group(3)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_cuenta",
                    "group_code": code,
                    "metric": "count",
                    "stat": k,
                    "description": f"第二板块：近{w}天内 tipoCuenta={code} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                }
            )
            continue

        # 大白话：tipoCuenta 指标统计。
        m = re.match(r"^creditos_(\d+)d_tipo_cuenta_([a-z]+)_([a-z0-9_]+)_(mean|std|min|max|notnull_ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            metric = m.group(3)
            stat = m.group(4)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_cuenta",
                    "group_code": code,
                    "metric": metric,
                    "stat": stat,
                    "description": f"第二板块：近{w}天内 tipoCuenta={code} 的 {metric} 的 {stat}",
                }
            )
            continue

        # 大白话：tipoCredito 计数/占比。
        m = re.match(r"^creditos_(\d+)d_tipo_credito_([a-z]+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            k = m.group(3)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_credito",
                    "group_code": code,
                    "metric": "count",
                    "stat": k,
                    "description": f"第二板块：近{w}天内 tipoCredito={code} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                }
            )
            continue

        # 大白话：tipoCredito 指标统计。
        m = re.match(r"^creditos_(\d+)d_tipo_credito_([a-z]+)_([a-z0-9_]+)_(mean|std|min|max|notnull_ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            metric = m.group(3)
            stat = m.group(4)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_credito",
                    "group_code": code,
                    "metric": metric,
                    "stat": stat,
                    "description": f"第二板块：近{w}天内 tipoCredito={code} 的 {metric} 的 {stat}",
                }
            )
            continue

        # 大白话：机构大类计数/占比（现在用英文ID做列名，比如 shop/bank/sic，不再出现中文）。
        m = re.match(r"^creditos_(\d+)d_(.+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            g = m.group(2)
            k = m.group(3)
            if g.startswith("tipo_"):
                pass
            else:
                g_cn = GROUP_ID_TO_NAME.get(g, g)  # 英文ID -> 中文名（用于描述）
                rows.append(
                    {
                        "block": "block2_creditos",
                        "feature_name": col,
                        "window_days": w,
                        "group_type": "otorgante_group",
                        "group_code": g,
                        "metric": "count",
                        "stat": k,
                        "description": f"第二板块：近{w}天内 机构大类={g_cn} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                    }
                )
                continue

        # 大白话：机构大类指标统计：creditos_{w}d_{groupId}_{metric}_{stat}
        m = re.match(r"^creditos_(\d+)d_(.+)_([a-z0-9_]+)_(mean|std|min|max|notnull_ratio)$", col)
        if m:
            w = int(m.group(1))
            g = m.group(2)
            metric = m.group(3)
            stat = m.group(4)
            if g.startswith("tipo_"):
                pass
            else:
                g_cn = GROUP_ID_TO_NAME.get(g, g)
                rows.append(
                    {
                        "block": "block2_creditos",
                        "feature_name": col,
                        "window_days": w,
                        "group_type": "otorgante_group",
                        "group_code": g,
                        "metric": metric,
                        "stat": stat,
                        "description": f"第二板块：近{w}天内 机构大类={g_cn} 的 {metric} 的 {stat}",
                    }
                )
                continue

        # 大白话：兜底：没解析到的列也记录下来，避免漏列。
        rows.append(
            {
                "block": "block2_creditos",
                "feature_name": col,
                "window_days": None,
                "group_type": "unknown",
                "group_code": None,
                "metric": None,
                "stat": None,
                "description": "第二板块：未匹配到解析规则（请人工确认列名口径）",
            }
        )

    # 大白话：输出为统一格式的数据字典表。
    return pd.DataFrame(
        rows,
        columns=[
            "block",
            "feature_name",
            "window_days",
            "group_type",
            "group_code",
            "metric",
            "stat",
            "description",
        ],
    )


# 大白话：生成并展示第二板块的特征字典。
feature_dict = build_feature_dict_block2(list(features.columns))
print("feature_dict shape:", feature_dict.shape)
feature_dict.head(10)

# === 导出：特征字典 CSV（3列：特征名称/中文释义/逻辑解释）===
# 大白话：你要的是每个板块各自导出一份“特征字典CSV”，用于你后续手工上下拼接。
# 大白话：默认不写文件：你确认后把 WRITE_FEATURE_DICT_OUTPUTS 改成 True 才会写出。
WRITE_FEATURE_DICT_OUTPUTS = False

# 大白话：第二板块涉及的“原始字段 -> 中文含义”，用于 logic_desc 解释字段名。
RAW_FIELD_DESC_BLOCK2 = {
    "response_body.creditos": "信贷账户列表",
    "fechaAperturaCuenta": "账户开立日期（开户日期，窗口锚点）",
    "request_time": "截止日期（本次申请时间，来自 apply_time 映射）",
    "nombreOtorgante": "机构类别（原始机构字符串）",
    "tipoCuenta": "账户类型（F/H/R/Q/A/E/P）",
    "tipoCredito": "信用/贷款类型（CC/PP/TC/LC/PB）",
    "montoPagar": "当期应还金额",
    "saldoActual": "当前总待还金额",
    "limiteCredito": "信用额度",
    "saldoVencido": "逾期余额/逾期金额",
    "valorActivoValuacion": "资产估值金额",
    "creditoMaximo": "历史最高授信额",
    "numeroPagos": "还款次数",
    "peorAtraso": "最严重逾期期数",
    "saldoVencidoPeorAtraso": "最严重逾期时的逾期金额",
    "fechaUltimoPago": "最后还款日期",
    "fechaCierreCuenta": "账户关闭日期",
    "fechaPeorAtraso": "最严重逾期日期",
}

# 大白话：metric（代码里的英文ID）对应哪个原始字段。
METRIC_TO_RAW_FIELD = {
    "monto_pagar": "montoPagar",
    "saldo_actual": "saldoActual",
    "limite_credito": "limiteCredito",
    "saldo_vencido": "saldoVencido",
    "valor_activo_valuacion": "valorActivoValuacion",
    "credito_maximo": "creditoMaximo",
    "numero_pagos": "numeroPagos",
    "peor_atraso": "peorAtraso",
    "saldo_vencido_peor_atraso": "saldoVencidoPeorAtraso",
    "days_since_last_payment": "fechaUltimoPago",
    "days_since_open": "fechaAperturaCuenta",
    "days_since_close": "fechaCierreCuenta",
    "is_closed": "fechaCierreCuenta",
    "days_pay_since_open": "fechaUltimoPago/fechaAperturaCuenta",
    "days_worst_arrears_since_open": "fechaPeorAtraso/fechaAperturaCuenta",
    "days_since_worst_arrears_date": "fechaPeorAtraso",
    "count": "record_count",
}

STAT_DESC_BLOCK2 = {
    "cnt": "数量（该组记录数）",
    "ratio": "占比=该组cnt/窗口内总记录数",
    "mean": "均值=mean(指标值)",
    "std": "标准差=std(指标值)",
    "min": "最小值=min(指标值)",
    "max": "最大值=max(指标值)",
    "notnull_ratio": "非空占比=该指标非空条数/该组总条数",
}

def _logic_desc_block2(r) -> str:
    w = r.get("window_days")
    gtype = r.get("group_type")
    gcode = r.get("group_code")
    metric = r.get("metric")
    stat = r.get("stat")

    # 窗口口径
    window_desc = (
        f"窗口={w}天（锚点fechaAperturaCuenta=账户开立日期；days_before_request_credit=(request_time-fechaAperturaCuenta)/86400；取0<=days<=window）"
    )

    # 分组来源字段说明
    group_desc = (
        f"分组={gtype}:{gcode}（来源字段：nombreOtorgante=机构类别 / tipoCuenta=账户类型 / tipoCredito=信用类型）"
    )

    # 指标来源字段说明
    raw_field = METRIC_TO_RAW_FIELD.get(str(metric), None)
    if raw_field and raw_field in RAW_FIELD_DESC_BLOCK2:
        metric_src = f"指标={metric}（来自{raw_field}={RAW_FIELD_DESC_BLOCK2[raw_field]}）"
    else:
        metric_src = f"指标={metric}"

    stat_explain = STAT_DESC_BLOCK2.get(str(stat), str(stat))

    raw_fields_glossary = (
        "原始字段释义："
        "request_time=截止日期；"
        "fechaAperturaCuenta=开户日期；"
        "montoPagar=当期应还金额；saldoActual=当前总待还金额；limiteCredito=信用额度；saldoVencido=逾期金额；"
        "valorActivoValuacion=资产估值；creditoMaximo=历史最大额度；numeroPagos=还款次数；peorAtraso=最严重逾期期数；saldoVencidoPeorAtraso=最严重逾期金额；"
        "fechaUltimoPago=最后还款日期；fechaCierreCuenta=账户关闭日期；fechaPeorAtraso=最严重逾期日期"
    )

    return f"{window_desc}; {group_desc}; {metric_src}; 统计={stat}（{stat_explain}）; {raw_fields_glossary}"

feature_dict_3col = pd.DataFrame(
    {
        "feature_name": feature_dict["feature_name"],
        "cn_desc": feature_dict["description"],
        "logic_desc": feature_dict.apply(_logic_desc_block2, axis=1),
    }
)

feature_dict_out_path = OUTPUT_DIR / "feature_dict_creditos.csv"

if WRITE_FEATURE_DICT_OUTPUTS:
    feature_dict_3col.to_csv(feature_dict_out_path, index=False, encoding="utf-8-sig")
    print("written feature_dict:")
    print("-", feature_dict_out_path)
else:
    print("[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出特征字典CSV（等你确认后再输出）")

# === 特征列名统一加前后缀（只改 features，不改特征字典）===
FEATURE_PREFIX = "cdc_"
FEATURE_SUFFIX = "_v2"

rename_map = {
    c: f"{FEATURE_PREFIX}{c}{FEATURE_SUFFIX}"
    for c in features.columns
    if c != "request_time"
}
features = features.rename(columns=rename_map)

# === 衍生特征统一保留2位小数（只改 features，不改特征字典；request_time 不动）===
_round_cols = [c for c in features.columns if c != "request_time"]
features[_round_cols] = features[_round_cols].apply(pd.to_numeric, errors="coerce").round(2)

# 大白话：你要求“打印 features 的所有行”。
# 大白话：这里把 pandas 的显示行数放开（不截断行），但列数仍按一定上限显示（否则 1.6w+ 列会让输出不可读/非常慢）。
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# 大白话：打印 features（所有行；列会按 max_columns 自动省略）。
features


[INFO] 当前 WRITE_OUTPUTS=False：已计算完成，但未写出 outputs/*.csv（等你确认后再输出）
creditos_flat rows: 697815
features shape: (12546, 16647)
features.request_time NaT rate: 0.0
features.request_time head: [Timestamp('2025-11-24 21:37:16.301000'), Timestamp('2025-11-24 21:41:55.655000'), Timestamp('2025-11-24 21:43:14.452000')]
feature_dict shape: (16646, 8)
[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出特征字典CSV（等你确认后再输出）


,request_time,cdc_creditos_30d_shop_cnt_v2,cdc_creditos_30d_shop_ratio_v2,cdc_creditos_30d_mass_fin_assn_cnt_v2,cdc_creditos_30d_mass_fin_assn_ratio_v2,cdc_creditos_30d_multi_purpose_cnt_v2,cdc_creditos_30d_multi_purpose_ratio_v2,cdc_creditos_30d_nonbank_mortgage_cnt_v2,cdc_creditos_30d_nonbank_mortgage_ratio_v2,cdc_creditos_30d_service_cnt_v2,cdc_creditos_30d_service_ratio_v2,cdc_creditos_30d_paytv_cnt_v2,cdc_creditos_30d_paytv_ratio_v2,cdc_creditos_30d_personal_loan_cnt_v2,cdc_creditos_30d_personal_loan_ratio_v2,cdc_creditos_30d_fund_trust_cnt_v2,cdc_creditos_30d_fund_trust_ratio_v2,cdc_creditos_30d_construction_cnt_v2,cdc_creditos_30d_construction_ratio_v2,cdc_creditos_30d_finance_company_cnt_v2,cdc_creditos_30d_finance_company_ratio_v2,cdc_creditos_30d_telecom_cnt_v2,cdc_creditos_30d_telecom_ratio_v2,cdc_creditos_30d_catalog_sale_cnt_v2,cdc_creditos_30d_catalog_sale_ratio_v2,cdc_creditos_30d_micro_loan_cnt_v2,cdc_creditos_30d_micro_loan_ratio_v2,cdc_creditos_30d_bank_cnt_v2,cdc_creditos_30d_bank_ratio_v2,cdc_creditos_30d_nonbank_cnt_v2,...,cdc_creditos_720d_tipo_credito_pb_days_worst_arrears_since_open_mean_v2,cdc_creditos_720d_tipo_credito_pb_days_worst_arrears_since_open_std_v2,cdc_creditos_720d_tipo_credito_pb_days_worst_arrears_since_open_min_v2,cdc_creditos_720d_tipo_credito_pb_days_worst_arrears_since_open_max_v2,cdc_creditos_720d_tipo_credito_pb_days_worst_arrears_since_open_notnull_ratio_v2,cdc_creditos_720d_tipo_credito_cc_days_since_worst_arrears_date_mean_v2,cdc_creditos_720d_tipo_credito_cc_days_since_worst_arrears_date_std_v2,cdc_creditos_720d_tipo_credito_cc_days_since_worst_arrears_date_min_v2,cdc_creditos_720d_tipo_credito_cc_days_since_worst_arrears_date_max_v2,cdc_creditos_720d_tipo_credito_cc_days_since_worst_arrears_date_notnull_ratio_v2,cdc_creditos_720d_tipo_credito_pp_days_since_worst_arrears_date_mean_v2,cdc_creditos_720d_tipo_credito_pp_days_since_worst_arrears_date_std_v2,cdc_creditos_720d_tipo_credito_pp_days_since_worst_arrears_date_min_v2,cdc_creditos_720d_tipo_credito_pp_days_since_worst_arrears_date_max_v2,cdc_creditos_720d_tipo_credito_pp_days_since_worst_arrears_date_notnull_ratio_v2,cdc_creditos_720d_tipo_credito_tc_days_since_worst_arrears_date_mean_v2,cdc_creditos_720d_tipo_credito_tc_days_since_worst_arrears_date_std_v2,cdc_creditos_720d_tipo_credito_tc_days_since_worst_arrears_date_min_v2,cdc_creditos_720d_tipo_credito_tc_days_since_worst_arrears_date_max_v2,cdc_creditos_720d_tipo_credito_tc_days_since_worst_arrears_date_notnull_ratio_v2,cdc_creditos_720d_tipo_credito_lc_days_since_worst_arrears_date_mean_v2,cdc_creditos_720d_tipo_credito_lc_days_since_worst_arrears_date_std_v2,cdc_creditos_720d_tipo_credito_lc_days_since_worst_arrears_date_min_v2,cdc_creditos_720d_tipo_credito_lc_days_since_worst_arrears_date_max_v2,cdc_creditos_720d_tipo_credito_lc_days_since_worst_arrears_date_notnull_ratio_v2,cdc_creditos_720d_tipo_credito_pb_days_since_worst_arrears_date_mean_v2,cdc_creditos_720d_tipo_credito_pb_days_since_worst_arrears_date_std_v2,cdc_creditos_720d_tipo_credito_pb_days_since_worst_arrears_date_min_v2,cdc_creditos_720d_tipo_credito_pb_days_since_worst_arrears_date_max_v2,cdc_creditos_720d_tipo_credito_pb_days_since_worst_arrears_date_notnull_ratio_v2
apply_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1065462364007251969,2025-11-24 21:37:16.301,0,0.0,0,0.0,0,0.00,0,0.0,0,0.00,0,0.0,0,0.00,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,29.900000,23.900000,8.900000,55.900002,0.23,85.900002,NaN,85.900002,85.900002,0.20,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,0.0
1065471950374248449,2025-11-24 21:41:55.655,0,0.0,0,0.0,0,0.00,0,0.0,0,0.00,0,0.0,0,0.00,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,32.700001,22.590000,4.900000,73.900002,0.18,24.900000,NaN,24.900000,24.900000,1.00,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,0.0
1065474664793579521,2025-11-24 21:43:14.452,0,0.0,0,0.0,0,0.00,0,0.0,0,0

In [ ]:
# TODO（评估模块）
# 大白话：这一格专门做“效果评估/稳定性评估”的汇总输出：
# - 1）按周（week）统计：整体坏率、cycle_pass 坏率、single_pass 坏率
# - 2）整体 TopIV 特征：等频/等距 分 5 箱（缺失单独一箱），看每箱总量/坏样本/坏率/唯一值数
# - 3）cycle_pass / single_pass 各自 TopIV 特征：同样做等频/等距 5 箱
# - 4）IV>0.1 的特征表：iv、corr_fpd7、na_rate(空值或-999)、psi(后两周vs第一周)、iv_cycle_pass、iv_single_pass
# - 5）“分周 IV”：整体/ cycle_pass / single_pass 维度下，对 Top 特征做每周 IV 计算（用于看稳定性随时间漂移）
# 重要约定：
# - y = (fpd7 > 0) 视为坏样本=1，否则好样本=0
# - 空值定义：NaN 或 -999 都当作空值（不默认把 0 当空值）
# - 分周：用 features["request_time"]（=apply_time 映射而来）按周一对齐（week_start）
# - 默认不写任何文件；如果你确认要落盘，我留了开关 WRITE_REPORTS

# ========== 0. 依赖检查与基础对齐（防止一跑就报错） ==========
# 大白话：这里为了稳，先把常用库显式 import 一次（即使上面已经 import 过也不影响）。
import re
import numpy as np
import pandas as pd
from pathlib import Path

# 大白话：统一的“空值哨兵”值：A/C/D 兜底用 -999；评估里也把它当空值。
SENTINEL_VALUE = -999

# 大白话：衍生评估报告输出（Excel 多sheet）。你已经确认要“直接写进一个excel”。
WRITE_REPORTS = False
REPORT_XLSX = Path("outputs") / "block2_eval_report.xlsx"

# 大白话：一个小工具——安全除法，避免 0 做分母报错。
def _safe_div(a: float, b: float) -> float:
    # 大白话：b=0 时返回 NaN（代表无法计算）。
    return float(a) / float(b) if float(b) != 0 else float("nan")

# 大白话：确保 features 存在，并且索引是 apply_id。
# - 如果 features 里还有 apply_id 列，就把它设为 index。
# - request_time 必须存在且能解析为 datetime。
if "features" not in globals():
    raise RuntimeError("[EVAL] 没找到 features，请先运行上面的衍生结果代码格。")

if "apply_id" in features.columns:
    features = features.set_index("apply_id")

if "request_time" not in features.columns:
    raise RuntimeError("[EVAL] features 缺少 request_time，请先运行上面的衍生结果代码格。")

# 大白话：把 request_time 强制转成 datetime，解析失败会是 NaT。
features["request_time"] = pd.to_datetime(features["request_time"], errors="coerce")

# 大白话：确保 df_raw 存在，并且能拿到 fpd7/approve_state。
if "df_raw" not in globals():
    raise RuntimeError("[EVAL] 没找到 df_raw，请先运行第1格读取数据代码格。")

# 大白话：从 df_raw 抽取 apply_id -> fpd7，并对齐到 features.index。
# - y_bin：坏样本=1（fpd7>0），好样本=0（fpd7<=0）
_y = (
    df_raw[["apply_id", "fpd7"]]
    .drop_duplicates("apply_id")
    .set_index("apply_id")["fpd7"]
)
_y = pd.to_numeric(_y, errors="coerce")
_y01 = (_y > 0).astype("int8").reindex(features.index)

# 大白话：如果 approve_state 列不存在，就全部视为 ""（这样 cycle/single mask 会全 False）。
_state = (
    df_raw[["apply_id", "approve_state"]].drop_duplicates("apply_id").set_index("apply_id")["approve_state"]
    if "approve_state" in df_raw.columns
    else pd.Series("", index=df_raw["apply_id"].drop_duplicates().tolist(), dtype="object")
)
_state = _state.fillna("").astype(str).str.lower().reindex(features.index).fillna("")

# 大白话：两类样本的 mask。
mask_cycle = _state.str.contains("cycle_pass", na=False)
mask_single = _state.str.contains("single_pass", na=False)

# 大白话：用于分周的 week_start（周一 00:00:00）。
_t = features["request_time"]
week_start = (_t - pd.to_timedelta(_t.dt.weekday, unit="D")).dt.normalize()

# ========== 1）分周坏率：整体 / cycle_pass / single_pass ==========
# 大白话：分周坏率表 = 每周总量 / 坏样本量 / 坏率。
def _weekly_bad_rate(mask: pd.Series, name: str) -> pd.DataFrame:
    # 大白话：取子样本。
    idx = features.index[mask]
    # 大白话：组装周和 y，并过滤掉周为 NaT 的样本。
    tmp = pd.DataFrame({"week": week_start.reindex(idx), "y": _y01.reindex(idx)}).dropna(subset=["week", "y"])
    # 大白话：按周聚合。
    out = tmp.groupby("week", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
    # 大白话：坏率=坏样本/总样本。
    out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
    # 大白话：加一个标签列。
    out["sample"] = name
    return out

# 大白话：整体 mask（有 y 的都算；如果 request_time 为 NaT 会在函数里被 drop 掉）。
mask_all = pd.Series(True, index=features.index)

weekly_all = _weekly_bad_rate(mask_all, "overall")
weekly_cycle = _weekly_bad_rate(mask_cycle, "cycle_pass")
weekly_single = _weekly_bad_rate(mask_single, "single_pass")

weekly_bad_rate_df = pd.concat([weekly_all, weekly_cycle, weekly_single], axis=0, ignore_index=True)
weekly_bad_rate_df = weekly_bad_rate_df.sort_values(["week", "sample"], ascending=[True, True])

print("\n[WEEKLY_BAD_RATE] overall/cycle_pass/single_pass:")
print(weekly_bad_rate_df.to_string(index=False))

# ========== 2）基础统计：空值率/同一值率（用于筛选可算 IV 的特征） ==========
# 大白话：X = 所有衍生特征列（不含 request_time）。
X = features.drop(columns=["request_time"], errors="ignore").copy()

# 大白话：把 -999 也当空值。
missing_mask = X.isna() | (X == SENTINEL_VALUE)

# 大白话：每个特征的空值率。
na_rate = missing_mask.mean(axis=0)

# 大白话：同一值率 = 出现频率最高的值的占比（把缺失统一成一个占位符一起算）。
_same = {}
for c in X.columns:
    s = X[c].copy()
    s = s.mask(missing_mask[c], "__MISSING__")
    vc = s.value_counts(dropna=False, normalize=True)
    _same[c] = float(vc.iloc[0]) if len(vc) > 0 else 1.0
same_value_rate = pd.Series(_same).reindex(X.columns)

print("\n[PROFILE] feature_cnt:", int(X.shape[1]))
print("[PROFILE] na_rate>=0.9 cnt:", int((na_rate >= 0.9).sum()))
print("[PROFILE] same_value_rate>=0.99 cnt:", int((same_value_rate >= 0.99).sum()))

# ========== 3）IV 计算（用于 TopIV、IV>0.1 清单、分周IV） ==========
# 大白话：连续变量用分位数分箱（q=10）；缺失单独一箱；必要时退化为离散。
IV_QCUT_BINS = 10

# 大白话：单特征 IV。
def _compute_iv_one(x: pd.Series, y01: pd.Series) -> float:
    # 大白话：只保留 y 不为空的行。
    dfv = pd.DataFrame({"x": x, "y": y01}).dropna(subset=["y"])
    if dfv.shape[0] == 0:
        return float("nan")

    x_raw = dfv["x"]
    y = dfv["y"].astype(int)

    # 大白话：缺失定义：NaN 或 -999。
    miss = x_raw.isna() | (x_raw == SENTINEL_VALUE)
    x_non = x_raw[~miss]

    # 大白话：有效值太少或几乎常量：直接按取值离散分箱。
    if x_non.nunique(dropna=True) <= 2:
        b = x_raw.astype(object).where(~miss, "MISSING")
    else:
        # 大白话：连续变量：分位数分箱；失败则退化为离散。
        try:
            b_non = pd.qcut(x_non, q=IV_QCUT_BINS, duplicates="drop")
            b = pd.Series("MISSING", index=x_raw.index, dtype="object")
            b.loc[~miss] = b_non.astype(str)
        except Exception:
            b = x_raw.astype(object).where(~miss, "MISSING")

    # 大白话：按箱统计好/坏。
    grp = pd.DataFrame({"b": b, "y": y}).groupby("b", observed=False)["y"].agg(["count", "sum"]).rename(columns={"sum": "bad"})
    grp["good"] = grp["count"] - grp["bad"]

    bad_total = grp["bad"].sum()
    good_total = grp["good"].sum()
    if bad_total == 0 or good_total == 0:
        return 0.0

    # 大白话：平滑，避免某箱 bad/good 为 0 导致 log 爆炸。
    k = grp.shape[0]
    grp["bad_dist"] = (grp["bad"] + 0.5) / (bad_total + 0.5 * k)
    grp["good_dist"] = (grp["good"] + 0.5) / (good_total + 0.5 * k)

    woe = np.log(grp["bad_dist"] / grp["good_dist"])
    iv = ((grp["bad_dist"] - grp["good_dist"]) * woe).sum()
    return float(iv)

# 大白话：对子样本计算 IV。
def _compute_iv_one_subset(x_full: pd.Series, y01_full: pd.Series, mask: pd.Series) -> float:
    return _compute_iv_one(x_full[mask], y01_full[mask])

# 大白话：为了可控运行时间，这里只对“非全空/非常量且空值率<0.9”的特征算 IV。
valid_for_iv = (na_rate < 0.9) & (same_value_rate < 0.99)
eligible_cols = X.columns[valid_for_iv].tolist()

# 大白话：如果你担心运行太慢，可以设置一个上限（比如 3000）；None 表示全量 eligible。
TOP_SEARCH_MAX_FEATURES = None
if TOP_SEARCH_MAX_FEATURES is not None:
    eligible_cols = eligible_cols[: int(TOP_SEARCH_MAX_FEATURES)]

print("\n[IV] eligible feature cnt:", int(len(eligible_cols)), "/ total:", int(X.shape[1]))

# 大白话：计算整体 IV（用于找整体 Top1）。
iv_overall = {}
for c in eligible_cols:
    iv_overall[c] = _compute_iv_one(pd.to_numeric(X[c], errors="coerce"), _y01)
iv_overall_s = pd.Series(iv_overall, name="iv_overall")

# 大白话：整体 Top1。
feat_top_overall = str(iv_overall_s.sort_values(ascending=False).index[0]) if len(iv_overall_s) > 0 else None
print("[IV] top1 overall:", feat_top_overall, "iv=", float(iv_overall_s.max()) if len(iv_overall_s) > 0 else None)

# 大白话：cycle_pass Top1（在 cycle 子样本里重新算 IV 再取最大）。
iv_cycle = {}
if int(mask_cycle.sum()) > 0:
    for c in eligible_cols:
        iv_cycle[c] = _compute_iv_one_subset(pd.to_numeric(X[c], errors="coerce"), _y01, mask_cycle)
    iv_cycle_s = pd.Series(iv_cycle, name="iv_cycle_pass")
    feat_top_cycle = str(iv_cycle_s.sort_values(ascending=False).index[0]) if len(iv_cycle_s) > 0 else None
else:
    iv_cycle_s = pd.Series(dtype="float64", name="iv_cycle_pass")
    feat_top_cycle = None
print("[IV] top1 cycle_pass:", feat_top_cycle, "iv=", float(iv_cycle_s.max()) if len(iv_cycle_s) > 0 else None)

# 大白话：single_pass Top1。
iv_single = {}
if int(mask_single.sum()) > 0:
    for c in eligible_cols:
        iv_single[c] = _compute_iv_one_subset(pd.to_numeric(X[c], errors="coerce"), _y01, mask_single)
    iv_single_s = pd.Series(iv_single, name="iv_single_pass")
    feat_top_single = str(iv_single_s.sort_values(ascending=False).index[0]) if len(iv_single_s) > 0 else None
else:
    iv_single_s = pd.Series(dtype="float64", name="iv_single_pass")
    feat_top_single = None
print("[IV] top1 single_pass:", feat_top_single, "iv=", float(iv_single_s.max()) if len(iv_single_s) > 0 else None)

# ========== 4）TopIV 特征：等频/等距 5 箱（缺失单独一箱） ==========
# 大白话：你要的输出：每箱总量、坏样本量、坏率、唯一值数量。
BIN_N = 5

# 大白话：把一个特征做“分箱 + 统计”的通用函数。
def _bin_and_report(feat: str, mask: pd.Series, method: str) -> pd.DataFrame:
    # 大白话：取子样本的 x/y。
    x = pd.to_numeric(X[feat], errors="coerce")
    y = _y01.reindex(x.index)
    x = x[mask]
    y = y[mask]

    # 大白话：缺失定义：NaN 或 -999。
    miss = x.isna() | (x == SENTINEL_VALUE)
    x_non = x[~miss]

    # 大白话：先构造 bin_label（缺失单独一个箱：MISSING）。
    if x_non.shape[0] == 0:
        bin_label = pd.Series("MISSING", index=x.index, dtype="object")
    else:
        if method == "qcut":
            # 大白话：等频：为了更稳定地得到 5 箱，用 rank() 处理重复值（缺失不参与）。
            r = x_non.rank(method="first")
            try:
                b_non = pd.qcut(r, q=BIN_N, duplicates="drop")
                bin_label = pd.Series("MISSING", index=x.index, dtype="object")
                bin_label.loc[~miss] = b_non.astype(str)
            except Exception:
                # 大白话：兜底：失败就按取值离散。
                bin_label = x.astype(object).where(~miss, "MISSING")
        elif method == "cut":
            # 大白话：等距：在非缺失的 min-max 上切 5 段。
            try:
                b_non = pd.cut(x_non, bins=BIN_N, include_lowest=True)
                bin_label = pd.Series("MISSING", index=x.index, dtype="object")
                bin_label.loc[~miss] = b_non.astype(str)
            except Exception:
                bin_label = x.astype(object).where(~miss, "MISSING")
        else:
            raise ValueError("unknown method: " + str(method))

    # 大白话：汇总每箱总量/坏样本/坏率。
    box = pd.DataFrame({"bin": bin_label, "y": y}).dropna(subset=["y"])
    out = box.groupby("bin", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
    out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)

    # 大白话：唯一值数量：只统计真实非缺失取值；缺失箱记 0。
    x_for_uniq = x.where(~miss, np.nan)
    uniq = (
        pd.DataFrame({"bin": bin_label, "x": x_for_uniq})
        .groupby("bin", observed=False)["x"]
        .nunique(dropna=True)
        .rename("unique_cnt")
        .reset_index()
    )
    out = out.merge(uniq, on="bin", how="left")
    out["unique_cnt"] = out["unique_cnt"].fillna(0).astype(int)

    # 大白话：排序：把 MISSING 放最上面，其余按 bin 字符串。
    out["_is_missing"] = (out["bin"] == "MISSING").astype(int)
    out = out.sort_values(["_is_missing", "bin"], ascending=[False, True]).drop(columns=["_is_missing"])

    # 大白话：补充 meta 信息。
    out.insert(0, "feature", feat)
    out.insert(1, "method", method)
    return out

# 大白话：分别打印整体 TopIV、cycle TopIV、single TopIV 的 5 箱结果。
# 大白话：为了把评估结果写进一个 Excel，我们把分箱结果也存成 DataFrame（不仅仅打印）。
BIN_TABLES: dict[str, pd.DataFrame] = {}

# 大白话：打印 + 返回两张表（qcut/cut），并放进 BIN_TABLES。
def _print_top_feat_bins(tag: str, feat: str, mask: pd.Series):
    if not feat:
        print(f"[BIN] {tag}: feat is None, skipped")
        return None, None

    df_q = _bin_and_report(feat, mask, method="qcut")
    df_c = _bin_and_report(feat, mask, method="cut")

    BIN_TABLES[f"bin_{tag}_qcut"] = df_q
    BIN_TABLES[f"bin_{tag}_cut"] = df_c

    print(f"[BIN] {tag} | feature={feat} | equal-frequency(qcut) bins={BIN_N} + missing")
    print(df_q.to_string(index=False))
    print(f"[BIN] {tag} | feature={feat} | equal-width(cut) bins={BIN_N} + missing")
    print(df_c.to_string(index=False))

    return df_q, df_c



_print_top_feat_bins("overall", feat_top_overall, mask_all)
_print_top_feat_bins("cycle_pass", feat_top_cycle, mask_cycle)
_print_top_feat_bins("single_pass", feat_top_single, mask_single)

# ========== 5）IV>0.1 清单：iv/corr/na_rate/psi/iv_cycle/iv_single ==========
# 大白话：先取整体 IV（iv_overall_s）里 >0.1 的特征作为候选清单。
IV_THRESHOLD = 0.1
iv_gt = iv_overall_s[iv_overall_s > IV_THRESHOLD].sort_values(ascending=False)

print("\n[IV_GT] IV>0.1 cnt:", int(iv_gt.shape[0]))

# 大白话：corr_fpd7：用非缺失样本计算 Pearson；缺失口径=NaN 或 -999。
_fpd7_num = (
    df_raw[["apply_id", "fpd7"]].drop_duplicates("apply_id").set_index("apply_id")["fpd7"]
    .pipe(pd.to_numeric, errors="coerce")
    .reindex(features.index)
)

# 大白话：PSI：后两周 vs 第一周；缺失单独一箱；非缺失用第一周的分位数边界分箱。
PSI_Q = 10
EPS = 1e-6

_weeks_sorted = sorted([w for w in week_start.dropna().unique()])
_first_w = _weeks_sorted[0] if len(_weeks_sorted) >= 1 else None
_last_two = _weeks_sorted[-2:] if len(_weeks_sorted) >= 2 else _weeks_sorted
_base_mask = (week_start == _first_w) if _first_w is not None else pd.Series(False, index=features.index)
_comp_mask = week_start.isin(_last_two) if len(_last_two) > 0 else pd.Series(False, index=features.index)

print("\n[PSI] first_week_start:", _first_w)
print("[PSI] last_two_week_start:", _last_two)
print("[PSI] base_cnt:", int(_base_mask.sum()), "comp_cnt:", int(_comp_mask.sum()))

# 大白话：单特征 PSI。
def _psi_one(x: pd.Series, base_mask: pd.Series, comp_mask: pd.Series) -> float:
    # 大白话：缺失定义：NaN 或 -999。
    miss = x.isna() | (x == SENTINEL_VALUE)

    xb = x[base_mask]
    xc = x[comp_mask]
    mb = miss[base_mask]
    mc = miss[comp_mask]

    # 大白话：如果 base 或 comp 没样本，返回 NaN。
    if xb.shape[0] == 0 or xc.shape[0] == 0:
        return float("nan")

    xb_non = xb[~mb]
    xc_non = xc[~mc]

    # 大白话：base 非缺失值太少：退化为离散箱。
    if xb_non.nunique(dropna=True) <= 2:
        bb = xb.astype(object).where(~mb, "MISSING")
        bc = xc.astype(object).where(~mc, "MISSING")
    else:
        try:
            _, edges = pd.qcut(xb_non, q=PSI_Q, retbins=True, duplicates="drop")
            edges = sorted(set(edges.tolist()))
            if len(edges) < 3:
                bb = xb.astype(object).where(~mb, "MISSING")
                bc = xc.astype(object).where(~mc, "MISSING")
            else:
                bb_non = pd.cut(xb_non, bins=edges, include_lowest=True)
                bc_non = pd.cut(xc_non, bins=edges, include_lowest=True)
                bb = pd.Series("MISSING", index=xb.index, dtype="object")
                bc = pd.Series("MISSING", index=xc.index, dtype="object")
                bb.loc[~mb] = bb_non.astype(str)
                bc.loc[~mc] = bc_non.astype(str)
        except Exception:
            bb = xb.astype(object).where(~mb, "MISSING")
            bc = xc.astype(object).where(~mc, "MISSING")

    pb = bb.value_counts(normalize=True)
    pc = bc.value_counts(normalize=True)

    cats = list(pb.index.union(pc.index))
    psi = 0.0
    for k in cats:
        p = float(pb.get(k, 0.0)); q = float(pc.get(k, 0.0))
        p = max(p, EPS); q = max(q, EPS)
        psi += (p - q) * float(np.log(p / q))
    return float(psi)

# 大白话：生成 IV>0.1 的报告表。
rows = []
for feat in iv_gt.index.tolist():
    x = pd.to_numeric(X[feat], errors="coerce")

    # 空值率（NaN 或 -999）
    miss = x.isna() | (x == SENTINEL_VALUE)
    na = float(miss.mean())

    # corr_fpd7（Pearson）：用非缺失且 fpd7 非空样本
    valid = (~miss) & _fpd7_num.notna()
    corr = float(pd.Series(x[valid]).corr(_fpd7_num[valid], method="pearson")) if int(valid.sum()) >= 3 else float("nan")

    # PSI（后两周 vs 第一周）
    psi = _psi_one(x, _base_mask, _comp_mask) if (_first_w is not None and len(_last_two) > 0) else float("nan")

    # 子样本 IV
    iv_c = _compute_iv_one_subset(x, _y01, mask_cycle) if int(mask_cycle.sum()) > 0 else float("nan")
    iv_sg = _compute_iv_one_subset(x, _y01, mask_single) if int(mask_single.sum()) > 0 else float("nan")

    rows.append(
        {
            "feature_name": feat,
            "iv": float(iv_gt.loc[feat]),
            "corr_fpd7": corr,
            "na_rate_na_or_-999": na,
            "psi_last2w_vs_firstw": psi,
            "iv_cycle_pass": iv_c,
            "iv_single_pass": iv_sg,
        }
    )

iv_report_df = pd.DataFrame(rows)

# 大白话：统一保留 4 位小数，便于阅读。
for c in ["iv", "corr_fpd7", "na_rate_na_or_-999", "psi_last2w_vs_firstw", "iv_cycle_pass", "iv_single_pass"]:
    if c in iv_report_df.columns:
        iv_report_df[c] = pd.to_numeric(iv_report_df[c], errors="coerce").round(4)

iv_report_df = iv_report_df.sort_values("iv", ascending=False).reset_index(drop=True)

print("\n[IV_REPORT] IV>0.1 report (head 50):")
print(iv_report_df.head(50).to_string(index=False))

# 大白话：如果你确认需要落盘，这里才写文件。
if WRITE_REPORTS:
    out_csv = Path("outputs") / f"block2_iv_gt_{IV_THRESHOLD}_report.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    iv_report_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[WRITE]", str(out_csv.resolve()))

# ========== 6）分周 IV：整体 / cycle_pass / single_pass（默认只算 Top 特征） ==========
# 大白话：你要“IV 值也按周算”，这里做每周单独计算该周样本上的 IV。
# 注意：某些周如果样本太少/全好或全坏，IV 会变成 0 或 NaN，这是正常的。

def _weekly_iv_for_feature(feat: str, mask: pd.Series) -> pd.DataFrame:
    x = pd.to_numeric(X[feat], errors="coerce")
    y = _y01.reindex(x.index)

    # 大白话：取子样本。
    idx = features.index[mask]
    w = week_start.reindex(idx)
    xv = x.reindex(idx)
    yv = y.reindex(idx)

    tmp = pd.DataFrame({"week": w, "x": xv, "y": yv}).dropna(subset=["week", "y"])

    # 大白话：逐周算 IV（每周都是独立的 x/y 分布）。
    out_rows = []
    for wk, g in tmp.groupby("week", observed=False):
        ivv = _compute_iv_one(g["x"], g["y"])
        out_rows.append({"week": wk, "iv": float(ivv), "total_cnt": int(g.shape[0]), "bad_cnt": int(g["y"].sum())})

    out = pd.DataFrame(out_rows).sort_values("week")
    out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
    out["iv"] = pd.to_numeric(out["iv"], errors="coerce").round(4)
    return out

# 大白话：需要做分周 IV 的特征列表（默认 3 个：整体Top、cycleTop、singleTop）。
weekly_iv_targets = []
if feat_top_overall:
    weekly_iv_targets.append(("overall_top", feat_top_overall))
if feat_top_cycle:
    weekly_iv_targets.append(("cycle_top", feat_top_cycle))
if feat_top_single:
    weekly_iv_targets.append(("single_top", feat_top_single))

WEEKLY_IV_TABLES: dict[str, pd.DataFrame] = {}

for tag, feat in weekly_iv_targets:
    df_w_all = _weekly_iv_for_feature(feat, mask_all)
    df_w_cycle = _weekly_iv_for_feature(feat, mask_cycle)
    df_w_single = _weekly_iv_for_feature(feat, mask_single)

    WEEKLY_IV_TABLES[f"weekly_iv_{tag}_overall"] = df_w_all
    WEEKLY_IV_TABLES[f"weekly_iv_{tag}_cycle_pass"] = df_w_cycle
    WEEKLY_IV_TABLES[f"weekly_iv_{tag}_single_pass"] = df_w_single

    print(f"[WEEKLY_IV] {tag} | feature={feat} | sample=overall")
    print(df_w_all.to_string(index=False))
    print(f"[WEEKLY_IV] {tag} | feature={feat} | sample=cycle_pass")
    print(df_w_cycle.to_string(index=False))
    print(f"[WEEKLY_IV] {tag} | feature={feat} | sample=single_pass")
    print(df_w_single.to_string(index=False))

# 大白话：如果你想把“IV>0.1 全部特征”也做分周 IV，会非常慢；这里留一个开关。
WEEKLY_IV_FOR_ALL_IV_GT = False
WEEKLY_IV_MAX_FEATURES = 50  # 大白话：避免输出爆炸，最多前 N 个（按整体IV从高到低）

if WEEKLY_IV_FOR_ALL_IV_GT:
    feats = iv_gt.index.tolist()[: int(WEEKLY_IV_MAX_FEATURES)]
    for feat in feats:
        print(f"\n[WEEKLY_IV_ALL] feature={feat} | sample=overall")
        print(_weekly_iv_for_feature(feat, mask_all).to_string(index=False))
        print(f"\n[WEEKLY_IV_ALL] feature={feat} | sample=cycle_pass")
        print(_weekly_iv_for_feature(feat, mask_cycle).to_string(index=False))
        print(f"\n[WEEKLY_IV_ALL] feature={feat} | sample=single_pass")
        print(_weekly_iv_for_feature(feat, mask_single).to_string(index=False))


# ========== 7）写出一个 Excel：衍生评估报告（多 sheet） ==========
# 大白话：你要“最后这些评估结果直接写进一个excel作为衍生报告”。这里把本格所有核心表写到同一个 xlsx。

def _safe_sheet_name(name: str, used: set[str]) -> str:
    # 大白话：Excel sheet 名最长 31；且不能重复。
    base = re.sub(r"[^0-9a-zA-Z_]+", "_", str(name))[:31] or "sheet"
    s = base
    i = 1
    while s in used:
        suffix = f"_{i}"
        s = (base[: max(0, 31 - len(suffix))] + suffix)
        i += 1
    used.add(s)
    return s

if WRITE_REPORTS:
    REPORT_XLSX.parent.mkdir(parents=True, exist_ok=True)

    # 大白话：汇总要写入的表。
    tables: dict[str, pd.DataFrame] = {}
    tables["weekly_bad_rate"] = weekly_bad_rate_df

    # 分箱结果（3个：overall/cycle/single × qcut/cut）
    if "BIN_TABLES" in globals():
        tables.update(BIN_TABLES)

    # IV>0.1 报告表
    tables["iv_gt_0p1_report"] = iv_report_df

    # 分周 IV（Top 特征）
    if "WEEKLY_IV_TABLES" in globals():
        tables.update(WEEKLY_IV_TABLES)

    # 元信息
    meta = pd.DataFrame(
        {
            "item": [
                "y_definition",
                "missing_definition",
                "week_anchor",
                "bin_n",
                "iv_threshold",
                "top_overall_feature",
                "top_cycle_feature",
                "top_single_feature",
                "cycle_cnt",
                "single_cnt",
                "total_cnt",
            ],
            "value": [
                "y=(fpd7>0)->1 else 0",
                "missing = NaN or -999",
                "week_start = Monday 00:00:00 based on request_time",
                str(BIN_N),
                str(IV_THRESHOLD),
                str(feat_top_overall),
                str(feat_top_cycle),
                str(feat_top_single),
                str(int(mask_cycle.sum())),
                str(int(mask_single.sum())),
                str(int(features.shape[0])),
            ],
        }
    )

    # 大白话：写 Excel（优先 openpyxl）。
    used = set()
    try:
        writer = pd.ExcelWriter(REPORT_XLSX, engine="openpyxl")
    except Exception:
        writer = pd.ExcelWriter(REPORT_XLSX)

    with writer:
        meta.to_excel(writer, sheet_name=_safe_sheet_name("meta", used), index=False)
        for k, df in tables.items():
            if df is None:
                df = pd.DataFrame()
            if not isinstance(df, pd.DataFrame):
                df = pd.DataFrame({"value": [str(df)]})
            df.to_excel(writer, sheet_name=_safe_sheet_name(k, used), index=False)

    print("[WRITE_EXCEL] eval report saved:", str(REPORT_XLSX.resolve()))

# =========================
# 额外：生成 quality 检测 Excel（每个特征一行）
# =========================
# 大白话：按你的要求，第二板块 notebook 也能独立输出 quality 表。
# 注意：第二板块特征列数非常多（1w+），打开 WRITE_QUALITY_EXCEL 后计算会很慢。

WRITE_QUALITY_EXCEL = False
QUALITY_XLSX = Path("outputs") / "block2_feature_quality.xlsx"

if WRITE_QUALITY_EXCEL:
    import scripts.build_all_blocks_feature_quality_excel as q

    _base = df_raw[["apply_id", "request_time", "fpd7", "approve_state"]].copy()
    _base["apply_id"] = _base["apply_id"].astype(str)
    _base = _base.drop_duplicates("apply_id").set_index("apply_id")

    _y = (pd.to_numeric(_base["fpd7"], errors="coerce") > 0).astype(int)
    _state = _base["approve_state"].fillna("").astype(str).str.lower()
    _is_cycle = _state.str.contains("cycle_pass", na=False)
    _is_single = _state.str.contains("single_pass", na=False)

    _rt = pd.to_datetime(_base["request_time"], errors="coerce")
    _week_start = (_rt - pd.to_timedelta(_rt.dt.weekday, unit="D")).dt.normalize()
    _weeks = sorted([w for w in _week_start.dropna().unique()])
    _first_w = _weeks[0] if len(_weeks) >= 1 else None
    _last_two = _weeks[-2:] if len(_weeks) >= 2 else _weeks
    _base_mask = (_week_start == _first_w) if _first_w is not None else pd.Series(False, index=_week_start.index)
    _comp_mask = _week_start.isin(_last_two) if len(_last_two) > 0 else pd.Series(False, index=_week_start.index)

    _feat = features.copy()
    if "apply_id" in _feat.columns:
        _feat = _feat.set_index("apply_id")
    _feat.index = _feat.index.astype(str)
    _X = _feat.drop(columns=["request_time"], errors="ignore")

    # 第二板块字典 feature_name 不带前后缀。
    _dict = feature_dict_3col.copy()
    _dict["feature_name"] = _dict["feature_name"].astype(str)
    _dict = _dict.drop_duplicates("feature_name").set_index("feature_name")

    _rows = []
    _total = int(_X.shape[0])

    for _col in _X.columns:
        _col_name = str(_col)
        _base_name = q._strip_feature_name(_col_name)

        _x = pd.to_numeric(_X[_col], errors="coerce")
        _miss = _x.isna() | (_x == q.SENTINEL)
        _na_cnt = int(_miss.sum())
        _na_rate = float(_na_cnt / _total) if _total else float("nan")
        _non = _x[~_miss]
        _nunique = int(_non.nunique(dropna=True))

        _cn = str(_dict["cn_desc"].get(_base_name, "")) if _base_name in _dict.index else ""
        _logic = str(_dict["logic_desc"].get(_base_name, "")) if _base_name in _dict.index else ""

        _corr = float("nan")
        _valid = (~_miss) & _y.notna()
        if int(_valid.sum()) >= 3:
            _corr = q._corr_pearson(_x[_valid].to_numpy(dtype="float64", copy=False), _y[_valid].to_numpy(dtype="float64", copy=False))

        _iv = q._iv_one(_x, _y)
        _psi = q._psi_one(_x, _base_mask, _comp_mask)
        _iv_c = q._iv_one(_x[_is_cycle], _y[_is_cycle])
        _iv_s = q._iv_one(_x[_is_single], _y[_is_single])

        _rows.append(
            {
                "feature_name": _base_name,
                "cn_desc": _cn,
                "logic_desc": _logic,
                "na_cnt": _na_cnt,
                "total_cnt": _total,
                "na_rate": round(_na_rate, 6),
                "nunique": _nunique,
                "corr_y": None if pd.isna(_corr) else round(float(_corr), 6),
                "iv": None if pd.isna(_iv) else round(float(_iv), 6),
                "psi_last2w_vs_firstw": None if pd.isna(_psi) else round(float(_psi), 6),
                "iv_cycle_pass": None if pd.isna(_iv_c) else round(float(_iv_c), 6),
                "iv_single_pass": None if pd.isna(_iv_s) else round(float(_iv_s), 6),
                "notebook": "第二板块衍生.ipynb",
            }
        )

    _quality = pd.DataFrame(_rows)
    QUALITY_XLSX.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(QUALITY_XLSX) as _w:
        _quality.to_excel(_w, sheet_name="report", index=False)
    print("[WRITE_EXCEL] block2 feature quality saved:", str(QUALITY_XLSX.resolve()))
